In [44]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/home-data-for-ml-course/sample_submission.csv
/kaggle/input/competitions/home-data-for-ml-course/sample_submission.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/train.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/data_description.txt
/kaggle/input/competitions/home-data-for-ml-course/test.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/train.csv
/kaggle/input/competitions/home-data-for-ml-course/test.csv


In [45]:
#Libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split


In [46]:
train = pd.read_csv('/kaggle/input/competitions/home-data-for-ml-course/train.csv')
train.sample()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
598,599,20,RL,80.0,12984,Pave,NaN,Reg,Bnk,AllPub,...,0,NaN,NaN,NaN,0,3,2006,WD,Normal,217500


In [47]:
train.shape

(1460, 81)

In [48]:
test = pd.read_csv('/kaggle/input/competitions/home-data-for-ml-course/test.csv')
print(test.shape)

(1459, 80)


In [49]:
test.sample(4)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
81,1542,50,RM,53.0,5830,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
80,1541,70,RM,60.0,10560,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
615,2076,20,RL,60.0,7200,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Abnorml
914,2375,120,RL,50.0,9466,Pave,NaN,IR2,Lvl,AllPub,...,217,0,NaN,NaN,NaN,0,5,2007,WD,Normal


**Here we can see some NaN values in some features means we have to clean this data first**

In [50]:
train.isnull().sum().sort_values(ascending=False)

PoolQC           1453
MiscFeature      1406
Alley            1369
Fence            1179
MasVnrType        872
                 ... 
MoSold              0
YrSold              0
SaleType            0
SaleCondition       0
SalePrice           0
Length: 81, dtype: int64

In [51]:
train.sample(4)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
796,797,20,RL,71.0,8197,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,4,2007,WD,Normal,143500
1015,1016,60,RL,70.0,8400,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,11,2009,WD,Normal,227000
738,739,90,RL,60.0,10800,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,3,2009,WD,Alloca,179000
431,432,50,RM,60.0,5586,Pave,NaN,IR1,Bnk,AllPub,...,0,NaN,MnPrv,NaN,0,9,2008,ConLD,Abnorml,79900


In [52]:
#Categorical 
train["MasVnrType"] = train["MasVnrType"].fillna('None')
test["MasVnrType"] = test["MasVnrType"].fillna('None')

train["Fence"] = train["Fence"].fillna('None')
test["Fence"] = test["Fence"].fillna('None')

#Numerical
Numer_cols = [
    "LotFrontage",
    "BsmtFullBath",
    "BsmtHalfBath",
    "TotalBsmtSF",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "GarageYrBlt",
    "MasVnrArea"
]
for col in Numer_cols:
    train[col] = train[col].fillna(train[col].median())
    test[col] = test[col].fillna(train[col].median())

#Dropping 
train.drop(columns=["PoolQC", "MiscFeature", "Alley"], inplace=True)
test.drop(columns=["PoolQC", "MiscFeature", "Alley"], inplace=True)


In [53]:
bsmt_cols = [
    "FireplaceQu",
    "GarageCond",
    "GarageType",
    "GarageQual",
    "GarageFinish",
    "BsmtQual",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "BsmtCond",
    "Electrical",
    "MSZoning",
    "Utilities",
    "Functional",
    "SaleType",
    "Exterior1st",
    "Exterior2nd",
    "KitchenQual"
]

for col in bsmt_cols:
    train[col] = train[col].fillna("None")
    test[col] = test[col].fillna("None")

In [54]:
test.isnull().sum().sort_values(ascending=False)

Id               0
MSSubClass       0
MSZoning         0
LotFrontage      0
LotArea          0
                ..
MiscVal          0
MoSold           0
YrSold           0
SaleType         0
SaleCondition    0
Length: 77, dtype: int64

In [55]:
train.isnull().sum().sort_values(ascending=False)

Id               0
MSSubClass       0
MSZoning         0
LotFrontage      0
LotArea          0
                ..
MoSold           0
YrSold           0
SaleType         0
SaleCondition    0
SalePrice        0
Length: 78, dtype: int64

In [56]:
test.shape

(1459, 77)

In [57]:
train.shape

(1460, 78)

In [58]:
print("SalePrice" in train.columns)
print("SalePrice" in test.columns)

True
False


In [59]:
X = train.drop("SalePrice",axis = 1)
y = train['SalePrice']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print(X_train.shape)
print(X_test.shape)

(1168, 77)
(292, 77)


In [60]:
X_train

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,Fence,MiscVal,MoSold,YrSold,SaleType,SaleCondition
254,255,20,RL,70.0,8400,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,0,None,0,6,2010,WD,Normal
1066,1067,60,RL,59.0,7837,Pave,IR1,Lvl,AllPub,Inside,...,0,0,0,0,None,0,5,2009,WD,Normal
638,639,30,RL,67.0,8777,Pave,Reg,Lvl,AllPub,Inside,...,164,0,0,0,MnPrv,0,5,2008,WD,Normal
799,800,50,RL,60.0,7200,Pave,Reg,Lvl,AllPub,Corner,...,264,0,0,0,MnPrv,0,6,2007,WD,Normal
380,381,50,RL,50.0,5000,Pave,Reg,Lvl,AllPub,Inside,...,242,0,0,0,None,0,5,2010,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1095,1096,20,RL,78.0,9317,Pave,IR1,Lvl,AllPub,Inside,...,0,0,0,0,None,0,3,2007,WD,Normal
1130,1131,50,RL,65.0,7804,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,0,MnPrv,0,12,2009,WD,Normal
1294,1295,20,RL,60.0,8172,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,0,None,0,4,2006,WD,Normal
860,861,50,RL,55.0,7642,Pave,Reg,Lvl,AllPub,Corner,...,0,0,0,0,GdPrv,0,6,2007,WD,Normal


In [61]:
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

cat_cols = X_train.select_dtypes(include ='object').columns
print(cat_cols)

Index(['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'Fence', 'SaleType', 'SaleCondition'],
      dtype='object')


In [62]:
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
print(num_cols)

Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd',
       'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF',
       'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea',
       'MiscVal', 'MoSold', 'YrSold'],
      dtype='object')


In [63]:
ohe = OneHotEncoder(
    handle_unknown ='ignore',
    sparse_output =False
)

**handle_unknown='ignore'
Prevents errors if the validation/test set contains a category not seen during training.**
**sparse_output=False
Returns a normal NumPy array instead of a sparse matrix.**

In [64]:
preprocessor = ColumnTransformer(
    transformers =[
        ("cat",ohe,cat_cols),
    ("num",StandardScaler(),num_cols)],
)

X_train_trf = preprocessor.fit_transform(X_train)
X_test_trf = preprocessor.transform(X_test)
test_trf = preprocessor.transform(test)

In [65]:
print(X_train_trf.shape)
print(X_test_trf.shape)
print(test_trf.shape)

(1168, 290)
(292, 290)
(1459, 290)


In [66]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train_trf,y_train)

LinearRegression()

In [67]:
y_pred  = lr.predict(X_test_trf)

In [68]:
from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(y_test,y_pred)
print(rmse)

29638.42923123804


In [69]:
from sklearn.metrics import r2_score
r2 = r2_score(y_test,y_pred)
print(r2)

0.8854759936452238


In [70]:
import numpy as np
from sklearn.metrics import root_mean_squared_error

log_rmse = root_mean_squared_error(
    np.log1p(y_test),
    np.log1p(y_pred)
)
print(log_rmse)

0.1791397431978063


**Observation : Linear Regression is usually not the best model for this competition.**

In [71]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state =42)
dt.fit(X_train_trf,y_train)

dt_pred = dt.predict(X_test_trf)

In [72]:
print("Under Decision Tree")
rmse = root_mean_squared_error(y_test,dt_pred)
print(rmse)

r2 = r2_score(y_test,dt_pred)
print(r2)

log_rmse = root_mean_squared_error(
    np.log1p(y_test),
    np.log1p(dt_pred)
)
print(log_rmse)

Under Decision Tree
44801.58110001971
0.7383186040795884
0.23741392558716853


In [73]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train_trf, y_train)

rf_pred = rf.predict(X_test_trf)

In [74]:
print('Under Random Forest')
rmse = root_mean_squared_error(y_test,rf_pred)
print(rmse)

r2 = r2_score(y_test,rf_pred)
print(r2)

log_rmse = root_mean_squared_error(
    np.log1p(y_test),
    np.log1p(rf_pred)
)
print(log_rmse)

Under Random Forest
29151.921073508962
0.8892049074158284
0.15373222149458868


In [75]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(random_state = 42)
gbr.fit(X_train_trf,y_train)
gbr_pred = gbr.predict(X_test_trf)

In [76]:
print('Under Gradient Boosting')
rmse = root_mean_squared_error(y_test,gbr_pred)
print(rmse)

r2 = r2_score(y_test,gbr_pred)
print(r2)

log_rmse = root_mean_squared_error(
    np.log1p(y_test),
    np.log1p(gbr_pred)
)
print(log_rmse)

Under Gradient Boosting
27226.592328510913
0.9033564792683341
0.13644199928396783


**Here we can see Gradient Boosting Provided us the Best Performance among other** 

**Now we will try to hyperparameter tuning to make our model more enhaced and better performance**

In [77]:
#RandomizedSearchCV
from sklearn.model_selection import RandomizedSearchCV

params_grid ={
    'n_estimators':[100,200,300,500],
    'learning_rate':[0.01,0.03,0.05,0.1],
    'max_depth':[2,3,4,5],
    'min_samples_split':[2,5,10],
    'min_samples_leaf':[1,2,4],
    'subsample':[0.8,0.9,1.0],
    'max_features':['sqrt','log2',None]
}

gbr = GradientBoostingRegressor(random_state =42)
random_search = RandomizedSearchCV(
    estimator = gbr,
    param_distributions = params_grid,
    n_iter = 30,
    scoring = 'neg_root_mean_squared_error',
    cv=5,
    verbose=2,
    random_state = 42,
    n_jobs=-1
)

random_search.fit(X_train_trf,y_train)

print(f'Best Parameters : {random_search.best_params_}')
print(f"Best CV Score:{-random_search.best_score_}")
best_gbr = random_search.best_estimator_
print(best_gbr)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters : {'subsample': 0.9, 'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 4, 'learning_rate': 0.1}
Best CV Score:27359.61216218407
GradientBoostingRegressor(max_depth=4, max_features='sqrt', min_samples_split=5,
                          n_estimators=300, random_state=42, subsample=0.9)


In [78]:
gbr = GradientBoostingRegressor(
    learning_rate=0.1,
    max_depth=4,
    max_features='sqrt',
    min_samples_split=5,
    min_samples_leaf=1,
    n_estimators=300,
    subsample=0.9,
    random_state=42
)
gbr.fit(X_train_trf,y_train)

GradientBoostingRegressor(max_depth=4, max_features='sqrt', min_samples_split=5,
                          n_estimators=300, random_state=42, subsample=0.9)

In [79]:
y_pred = gbr.predict(X_test_trf)

In [80]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
log_rmse = np.sqrt(mean_squared_error(np.log1p(y_test), np.log1p(y_pred)))

print("RMSE:", rmse)
print("R2:", r2)
print("Log RMSE:", log_rmse)

RMSE: 29151.68126207169
R2: 0.8892067302679857
Log RMSE: 0.14659490214109686


**Hyperparameter tuning was performed using RandomizedSearchCV with 30 parameter combinations and 5-fold cross-validation. Although the tuned model achieved the best cross-validation score during the search, its performance on the held-out validation set (Log RMSE = 0.1466) was worse than the original GradientBoostingRegressor (Log RMSE = 0.1364). Therefore, the original Gradient Boosting model was selected as the final model for submission.**

In [83]:
X_full_trf = preprocessor.fit_transform(X)
test_trf = preprocessor.transform(test)

In [84]:
final_model = GradientBoostingRegressor(random_state = 42)
final_model.fit(X_full_trf,y)

GradientBoostingRegressor(random_state=42)

In [85]:
predictions = final_model.predict(test_trf)

[CV] END learning_rate=0.01, max_depth=4, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=300, subsample=1.0; total time=   0.5s
[CV] END learning_rate=0.05, max_depth=5, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=200, subsample=0.9; total time=   0.4s
[CV] END learning_rate=0.05, max_depth=5, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=200, subsample=0.9; total time=   0.4s
[CV] END learning_rate=0.05, max_depth=3, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=300, subsample=1.0; total time=   0.4s
[CV] END learning_rate=0.01, max_depth=3, max_features=log2, min_samples_leaf=1, min_samples_split=10, n_estimators=500, subsample=0.9; total time=   0.8s
[CV] END learning_rate=0.1, max_depth=3, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=500, subsample=0.9; total time=   0.8s
[CV] END learning_rate=0.1, max_depth=3, max_features=log2, min_samples

In [86]:
submission = pd.DataFrame({
    "Id":test['Id'],
    "SalePrice":predictions
})

submission.to_csv('submission.csv',index=False)